# チュートリアル3: 結晶生成

このチュートリアルでは、周期境界条件を持つ結晶構造の生成方法を学びます。

**所要時間**: 30分

**学習内容**:
- ASEデータベースを使用した結晶データ
- 分子特徴抽出
- マルチモーダル条件付け
- ユニットセルの学習
- CIFファイルのエクスポート

**前提知識**: チュートリアル1（基本的な分子生成）


## セットアップとインポート


In [ ]:
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
from ase import Atoms
from ase.db import connect
from crystal.models import CrystalDynamics
from crystal.conditioning import MolecularConditioning
from crystal.utils import CIFWriter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {device}')


## 1. ASEデータベースと結晶データ

ASE (Atomic Simulation Environment) データベースを使用して結晶データを管理します。

**結晶データの要素**:
- 原子座標（分数座標または直交座標）
- セルパラメータ（a, b, c, alpha, beta, gamma）
- 周期境界条件（PBC）
- 空間群情報


In [ ]:
# ASEデータベースの接続
db_path = 'ase.db'  # デフォルトのデータベース

if os.path.exists(db_path):
    db = connect(db_path)
    print(f'データベースを開きました: {db_path}')
    print(f'エントリ数: {len(db)}')
    
    # 最初のエントリを確認
    if len(db) > 0:
        row = db.get(1)
        atoms = row.toatoms()
        print(f'\n最初のエントリ:')
        print(f'  原子数: {len(atoms)}')
        print(f'  化学式: {atoms.get_chemical_formula()}')
        print(f'  セルパラメータ: {atoms.cell.cellpar()}')
        print(f'  PBC: {atoms.pbc}')
else:
    print(f'データベースが見つかりません: {db_path}')
    print('create_test_db.py を実行してデータベースを作成してください。')


## 2. 周期境界条件（PBC）

結晶は無限に繰り返される構造です。周期境界条件により、ユニットセルの境界で原子が連続的につながります。

**PBCの実装**:
- 最近接画像規約（Minimum Image Convention）
- 分数座標での計算
- セルベクトルの考慮


In [ ]:
# PBCの例
print('周期境界条件の概念:')
print('\nユニットセル内の原子座標は分数座標 [0, 1) で表現されます。')
print('座標が1を超える、または0未満になる場合、')
print('自動的に隣接するユニットセルに折り返されます。')
print('\n例:')
print('  分数座標 1.2 -> 0.2 (隣のセルに移動)')
print('  分数座標 -0.3 -> 0.7 (反対側のセルに移動)')
print('\nこれにより、結晶の無限繰り返し構造をモデル化できます。')


## 3. 分子特徴抽出

結晶中の分子から特徴を抽出し、条件付けに使用します。


In [ ]:
# 分子特徴の抽出例
print('分子特徴抽出の実装:')
print('\n1. 幾何学的特徴:')
print('   - 分子サイズ（慣性テンソル）')
print('   - 形状記述子（球形度、扁平度）')
print('   - 表面積と体積')
print('\n2. 化学的特徴:')
print('   - 原子タイプ分布')
print('   - 官能基の有無')
print('   - 電荷分布')
print('\n3. 対称性特徴:')
print('   - 点群対称性')
print('   - 極性軸の有無')
print('\nこれらの特徴を使用して、結晶構造を条件付けます。')


## 4. 結晶拡散モデルの構築

周期境界条件を考慮した拡散モデルを構築します。


In [ ]:
# 結晶モデルの設定
class CrystalArgs:
    def __init__(self):
        self.n_layers = 5
        self.nf = 128
        self.diffusion_steps = 1000
        self.diffusion_noise_schedule = 'polynomial_2'
        self.pbc = [True, True, True]  # x, y, z方向全てでPBC
        self.normalize_factors = [1, 4, 1]
        self.include_charges = True
        self.condition_on_molecule = True  # 分子特徴での条件付け
        self.condition_on_space_group = False  # 空間群条件（チュートリアル6で）

crystal_args = CrystalArgs()

print('結晶拡散モデルの設定:')
print(f'  レイヤー数: {crystal_args.n_layers}')
print(f'  隠れ層次元: {crystal_args.nf}')
print(f'  拡散ステップ: {crystal_args.diffusion_steps}')
print(f'  PBC: {crystal_args.pbc}')
print(f'  分子条件付け: {crystal_args.condition_on_molecule}')


## 5. 結晶のサンプリング

学習済みモデルを使用して結晶構造を生成します。


In [ ]:
# サンプリングのデモンストレーション
print('結晶サンプリングの手順:')
print('\n1. 分子特徴の準備')
print('   context = extract_molecular_features(molecule)')
print('\n2. セルパラメータの初期化')
print('   cell = initial_cell_parameters(volume, shape)')
print('\n3. 拡散サンプリング')
print('   x, h, cell = model.sample(n_samples, context, pbc=[True,True,True])')
print('\n4. 分数座標への変換')
print('   fractional_coords = to_fractional(x, cell)')
print('\n5. 結晶構造の構築')
print('   atoms = Atoms(symbols, fractional_coords, cell=cell, pbc=True)')
print('\n注意: 完全な実装には main_crystal.py を参照してください。')


## 6. セルパラメータの学習

ユニットセルのサイズと形状も学習します。

**セルの表現**:
- 3×3行列（セルベクトル）
- 6パラメータ (a, b, c, α, β, γ)
- Cholesky分解による制約付き最適化


In [ ]:
# セルパラメータの学習
print('ユニットセルの学習方法:')
print('\n1. セルベクトルのパラメータ化')
print('   - 下三角行列（Cholesky分解）')
print('   - 正定値性の保証')
print('\n2. 損失関数')
print('   - 原子座標の再構成誤差')
print('   - セルパラメータの再構成誤差')
print('   - 密度制約（オプション）')
print('\n3. 最適化')
print('   - 座標とセルを同時に最適化')
print('   - E(3)同変性の維持')
print('\n実装の詳細は crystal/models.py を参照してください。')


## 7. CIFファイルのエクスポート

生成された結晶構造をCIF (Crystallographic Information File) 形式で保存します。

**CIFファイルの内容**:
- セルパラメータ
- 原子座標（分数座標）
- 空間群情報
- 対称操作


In [ ]:
# CIFエクスポートの例
print('CIFファイルの作成:')
print('\nASEを使用したCIFエクスポート:')
print('\n```python')
print('from ase.io import write')
print('')
print('# Atomsオブジェクトの作成')
print('atoms = Atoms(')
print('    symbols=["C", "H", "H", "H", "H"],')
print('    scaled_positions=fractional_coords,  # 分数座標')
print('    cell=cell_vectors,')
print('    pbc=True')
print(')')
print('')
print('# CIFファイルに書き出し')
print('write("output.cif", atoms)')
print('```')
print('\nCIFファイルは、結晶構造可視化ソフトウェア（VESTA、Mercury等）')
print('で開くことができます。')


## 8. 結晶品質の評価

生成された結晶の品質を評価します。

**評価指標**:
- 密度の妥当性
- 最短原子間距離
- セルパラメータの物理的妥当性
- 空間群対称性の保持


In [ ]:
# 結晶品質評価
print('結晶品質の評価方法:')
print('\n1. 密度チェック')
print('   density = mass / volume')
print('   典型的な有機結晶: 1.0-1.5 g/cm³')
print('\n2. 原子間距離')
print('   min_distance = get_shortest_distance(atoms, pbc=True)')
print('   C-C結合: ~1.5 Å')
print('   van der Waals: ~3.0 Å')
print('\n3. セルの妥当性')
print('   - 体積が正')
print('   - 角度が物理的に妥当（0° < α,β,γ < 180°）')
print('\n4. 対称性検証')
print('   - spglib で空間群を判定')
print('   - 指定した空間群との一致')
print('\n詳細は validate_generated_crystals.py を参照してください。')


## まとめ

このチュートリアルでは、以下を学習しました:

✅ ASEデータベースを使用した結晶データ管理  
✅ 周期境界条件の概念と実装  
✅ 分子特徴抽出による条件付け  
✅ 結晶拡散モデルの構築  
✅ セルパラメータの学習  
✅ CIFファイルのエクスポート  
✅ 結晶品質の評価方法  

### 次のステップ

- **チュートリアル4**: 分子記述子とASE - カスタム条件付け
- **チュートリアル6**: 高度な結晶条件付け - 空間群と密度制御
- **実践**: 実際のデータセットで結晶生成を実行

### 実行例

```bash
# テストデータベースの作成
python create_test_db.py

# 結晶モデルの学習
python main_crystal.py --exp_name crystal_demo --n_epochs 100

# 結晶のサンプリング
python eval_sample.py --model_path outputs/crystal_demo
```

### 重要な原則

1. **PBC の厳密な扱い**: 周期境界を常に考慮
2. **物理的妥当性**: セルパラメータと密度の制約
3. **フォールバックなし**: 明示的なエラーハンドリング

---

**質問やフィードバックは、GitHubのIssuesでお寄せください。**
